# Locked-rotor current-loop analysis

Analyze the closed-loop transfer from the q-axis current reference to measured, unfiltered q-axis current. The pybind11 wrapper runs the C++ drive and scheduler at the configured operating point and identifies a discrete small-signal model from the selected input and output.

Select the project's Python environment as the notebook kernel. If the extension is missing, run these commands from the repository root with that environment active.

```sh
uv pip install pybind11
PYTHON="$(command -v python)" pmsmcpp/scripts/build_python.sh
```

In [ ]:
from pathlib import Path
import sys

root = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "pmsmcpp" / "MODULE.bazel").is_file()),
    None,
)
if root is None:
    raise FileNotFoundError("Start the notebook kernel inside the MotorSim repository.")
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

import control as ct
import matplotlib.pyplot as plt
import numpy as np

from motorsim.models.drive import DriveModel

config = root / "pmsmcpp/configs/simplefoc_locked_rotor_config.json"
drive = DriveModel(config)

print("Writable input ports:", drive.input_ports)
print("Available output ports:", drive.output_ports)

## Operating point and sampled model

The supplied configuration requests 2 A with the rotor locked. Select the reference source and current feedback port from the discovered names. The perturbation is in amperes.

In [ ]:
input_port = "current_reference.value"
output_port = "current_feedback.iq"
delta_a = 1e-3

system, point = drive.linearize(input_port, output_port, delta=delta_a)
nyquist_hz = 0.5 / system.dt

print(f"Reference: {point['input_value']:.6g} A")
print(f"Measured current: {point['output_value']:.6g} A")
print(f"Sample time: {system.dt * 1e6:.3g} µs")
print(f"Nyquist frequency: {nyquist_hz:g} Hz")
print(f"DC gain: {float(system.dcgain()):.6g} A/A")
system

## Pole locations

Poles strictly inside the unit circle indicate stability of the modes visible from this reference to this measurement. The identified model describes the local input/output channel around the selected operating point.

In [ ]:
poles = system.pole()
largest_radius = np.max(np.abs(poles))
print("Poles:", poles)
print(f"Largest pole magnitude: {largest_radius:.6f}")

angle = np.linspace(0, 2 * np.pi, 361)
fig, ax = plt.subplots(figsize=(5, 5), layout="constrained")
ax.plot(np.cos(angle), np.sin(angle), "--", color="0.5", label="Unit circle")
ax.scatter(poles.real, poles.imag, marker="x", s=65, label="Identified poles")
ax.set(xlabel="Real", ylabel="Imaginary", title="Discrete current-loop poles")
ax.set_aspect("equal")
ax.grid(alpha=0.3)
ax.legend()
plt.show()

## Frequency response

Plot the response to a small sinusoidal current-reference perturbation. The frequency axis ends below Nyquist. Compare with a model identified at half the perturbation amplitude to check sensitivity to the identification step size.

In [ ]:
half_step_system, _ = drive.linearize(input_port, output_port, delta=delta_a / 2)
frequency_hz = np.geomspace(1, 0.99 * nyquist_hz, 400)
omega = 2 * np.pi * frequency_hz

fig, (gain_ax, phase_ax) = plt.subplots(
    2, 1, sharex=True, figsize=(8, 5), layout="constrained"
)
for model, amplitude, style in (
    (system, delta_a, "-"),
    (half_step_system, delta_a / 2, "--"),
):
    magnitude, phase, _ = ct.frequency_response(model, omega)
    gain_ax.semilogx(
        frequency_hz, 20 * np.log10(magnitude), style,
        label=f"{amplitude * 1e3:g} mA perturbation",
    )
    phase_ax.semilogx(frequency_hz, np.rad2deg(np.unwrap(phase)), style)

gain_ax.set(ylabel="Gain (dB)", title="Current reference to measured q-axis current")
phase_ax.set(xlabel="Frequency (Hz)", ylabel="Phase (deg)")
for ax in (gain_ax, phase_ax):
    ax.grid(which="both", alpha=0.3)
gain_ax.legend()
plt.show()

print(
    f"Change in largest pole magnitude at half the perturbation: "
    f"{abs(largest_radius - np.max(np.abs(half_step_system.pole()))):.2e}"
)

## C++ current-reference chirp

Apply a 0.1 A peak logarithmic sweep from 0.5 Hz to 8.5 kHz around the settled 2 A q-axis reference. Estimate the closed-loop response from the ratio of the logged unfiltered current and reference spectra, using 1 Hz to 8 kHz to stay inside the sweep endpoints. The dashed curves show the step-identified model at the same operating point.

In [ ]:
from scipy.signal import chirp
from scipy.signal.windows import tukey

# Sweep at the controller sample rate, then let the response settle back to DC.
sweep_s, tail_s = 12.0, 0.25
amplitude_a = 0.1
dt = drive.sample_time
sweep_samples = round(sweep_s / dt)
sweep_time = np.arange(sweep_samples) * dt
perturbation = (
    amplitude_a * chirp(sweep_time, f0=0.5, f1=8500, t1=sweep_s, method="logarithmic")
    * tukey(sweep_samples, alpha=0.05)
)
reference_a = point["input_value"]
start_tick = round(0.25 / dt) * drive.sample_ticks
changes = [
    (start_tick + i * drive.sample_ticks, input_port, reference_a + offset)
    for i, offset in enumerate(perturbation)
]
changes.append((start_tick + sweep_samples * drive.sample_ticks, input_port, reference_a))
_, trace = drive.run(
    [input_port, output_port, "current_pi.vq"],
    start_tick=start_tick,
    samples=sweep_samples + round(tail_s / dt),
    changes=changes,
)



In [ ]:
trace

In [ ]:
trace_time = np.arange(len(trace)) * dt
trace_fig, (overview_ax, detail_ax) = plt.subplots(2, 1, figsize=(8, 5), layout="constrained")
detail = (trace_time >= 4.0) & (trace_time <= 4.5)
for ax, interval in ((overview_ax, slice(None)), (detail_ax, detail)):
    ax.plot(trace_time[interval], trace[interval, 0], label="Current reference")
    ax.plot(trace_time[interval], trace[interval, 1], label="Measured q-axis current")
    ax.set(xlabel="Time from sweep start (s)", ylabel="Current (A)")
    ax.grid(alpha=0.3)
overview_ax.set_title("C++ current chirp")
overview_ax.legend(loc="upper right")
detail_ax.set_title("Sweep detail")
plt.show()

# Divide the FFTs of the logged, baseline-subtracted output and input.
chirp_reference = trace[:, 0] - reference_a
chirp_current = trace[:, 1] - point["output_value"]
frequency_bins = np.fft.rfftfreq(len(trace), dt)
band = (frequency_bins >= 1) & (frequency_bins <= 8000)
input_spectrum = np.fft.rfft(chirp_reference)[band]
assert np.min(np.abs(input_spectrum)) > 1e-3 * np.max(np.abs(input_spectrum))
assert np.max(np.abs(chirp_current[-100:])) < 1e-8
chirp_frequency = frequency_bins[band]
response = np.fft.rfft(chirp_current)[band] / input_spectrum
model_response = system(np.exp(2j * np.pi * chirp_frequency * dt))

gain_db = 20 * np.log10(np.abs(response))
phase_deg = np.rad2deg(np.unwrap(np.angle(response)))
model_gain_db = 20 * np.log10(np.abs(model_response))
model_phase_deg = np.rad2deg(np.unwrap(np.angle(model_response)))
peak = np.argmax(gain_db)
crossing = np.flatnonzero(gain_db[peak:] <= -3)[0] + peak
bandwidth_hz = np.interp(
    -3, gain_db[crossing - 1:crossing + 1][::-1],
    chirp_frequency[crossing - 1:crossing + 1][::-1],
)

fig, (gain_ax, phase_ax) = plt.subplots(
    2, 1, sharex=True, figsize=(8, 5), layout="constrained"
)
gain_ax.semilogx(chirp_frequency, gain_db, label="C++ chirp")
gain_ax.semilogx(chirp_frequency, model_gain_db, "--", label="Step-identified model")
gain_ax.axhline(-3, color="0.6", linewidth=0.8)
phase_ax.semilogx(chirp_frequency, phase_deg)
phase_ax.semilogx(chirp_frequency, model_phase_deg, "--")
gain_ax.set(ylabel="Gain (dB)", title="Locked-rotor q-axis current tracking")
phase_ax.set(xlabel="Frequency (Hz)", ylabel="Phase (deg)")
for ax in (gain_ax, phase_ax):
    ax.grid(which="both", alpha=0.3)
gain_ax.legend()
plt.show()

print(f"−3 dB tracking bandwidth: {bandwidth_hz:.1f} Hz")
print(f"Peak gain: {gain_db[peak]:+.2f} dB at {chirp_frequency[peak]:.1f} Hz")
print(f"Phase at bandwidth: {np.interp(bandwidth_hz, chirp_frequency, phase_deg):.1f}°")
print(f"Maximum magnitude difference: {np.max(np.abs(gain_db - model_gain_db)):.3g} dB")
print(f"Maximum phase difference: {np.max(np.abs(phase_deg - model_phase_deg)):.3g}°")
print(f"Peak |vq|: {np.max(np.abs(trace[:, 2])):.3f} V")